# Cross-View Retrieval Demo

This notebook demonstrates cross-view embedding retrieval for Boulder County POIs.
Given a query embedding, find the nearest matches by cosine similarity.

In [ ]:
import json
import numpy as np
from pathlib import Path

## Load embeddings

The embeddings are stored as INT8 quantized vectors in JSONL format.
We dequantize them back to float and re-normalize.

In [ ]:
def load_embeddings(path):
    """Load INT8 embeddings from JSONL, dequantize to float."""
    embeddings, metadata = [], []
    with open(path) as f:
        for line in f:
            entry = json.loads(line)
            vec = np.array(entry['ugl_vector'], dtype=np.float32) / entry['ugl_scale']
            vec = vec / np.linalg.norm(vec)
            embeddings.append(vec)
            metadata.append(entry)
    return np.stack(embeddings), metadata


# Load 256-d embeddings (or swap for 128d / 64d)
embeddings, meta = load_embeddings('../data/embeddings_256d.jsonl')
print(f'Loaded {len(embeddings)} embeddings, shape: {embeddings.shape}')

## Load POI metadata

In [ ]:
with open('../data/poi_metadata.json') as f:
    pois = json.load(f)

# Index by POI ID for quick lookup
poi_by_id = {p['poi_gers_id']: p for p in pois}
print(f'{len(pois)} POIs loaded')

## Nearest-neighbor retrieval

Pick a query POI and find its nearest neighbors in embedding space.
Nearby embeddings should share visual or structural characteristics
across both aerial and street-level perspectives.

In [ ]:
def find_nearest(query_idx, embeddings, metadata, k=10):
    """Find k nearest neighbors by cosine similarity."""
    sims = embeddings @ embeddings[query_idx]
    top_k = np.argsort(-sims)[:k+1]  # +1 because query matches itself
    results = []
    for idx in top_k:
        if idx == query_idx:
            continue
        m = metadata[idx]
        poi = poi_by_id.get(m['poi_gers_id'], {})
        results.append({
            'rank': len(results) + 1,
            'name': m['name'],
            'category': poi.get('category', ''),
            'similarity': float(sims[idx]),
            'lat': m['entrance_lat'],
            'lon': m['entrance_lon'],
            'mapillary_url': f"https://www.mapillary.com/app/?pKey={poi['mapillary_ids'][0]}" if poi.get('mapillary_ids') else None,
        })
    return results


# Query: pick a restaurant
query_idx = next(i for i, m in enumerate(meta) if 'restaurant' in poi_by_id.get(m['poi_gers_id'], {}).get('category', '').lower())
query_poi = poi_by_id[meta[query_idx]['poi_gers_id']]
print(f"Query: {query_poi['name']} ({query_poi['category']})")
print(f"Mapillary: https://www.mapillary.com/app/?pKey={query_poi['mapillary_ids'][0]}")
print()

results = find_nearest(query_idx, embeddings, meta, k=10)
for r in results:
    print(f"  #{r['rank']}  sim={r['similarity']:.3f}  {r['name']} ({r['category']})")
    if r['mapillary_url']:
        print(f"       {r['mapillary_url']}")

## Compare embedding dimensions

Load the same POIs at different MRL truncation levels and see how
retrieval rankings change (or don't).

In [ ]:
dims = [64, 128, 256]
all_emb = {}

for d in dims:
    path = f'../data/embeddings_{d}d.jsonl'
    try:
        emb, _ = load_embeddings(path)
        all_emb[d] = emb
        print(f'{d}-d: loaded {len(emb)} embeddings')
    except FileNotFoundError:
        print(f'{d}-d: file not found, skipping')

In [ ]:
# Compare top-5 neighbors across dimensions for the same query
if len(all_emb) > 1:
    print(f"Query: {meta[query_idx]['name']}\n")
    for d, emb in sorted(all_emb.items()):
        sims = emb @ emb[query_idx]
        top5 = np.argsort(-sims)[1:6]
        print(f"  {d}-d top 5:")
        for rank, idx in enumerate(top5, 1):
            print(f"    #{rank} sim={sims[idx]:.3f}  {meta[idx]['name']}")
        print()

## Spatial visualization

Plot POI locations colored by embedding cluster.

In [ ]:
try:
    from sklearn.cluster import KMeans
    import matplotlib.pyplot as plt

    n_clusters = 12
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = km.fit_predict(embeddings)

    lats = [m['entrance_lat'] for m in meta]
    lons = [m['entrance_lon'] for m in meta]

    fig, ax = plt.subplots(figsize=(12, 8))
    scatter = ax.scatter(lons, lats, c=labels, cmap='tab20', s=8, alpha=0.7)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'Boulder County POIs colored by embedding cluster (k={n_clusters})')
    plt.colorbar(scatter, label='Cluster')
    plt.tight_layout()
    plt.show()

except ImportError:
    print('Install scikit-learn and matplotlib for visualization:')
    print('  pip install scikit-learn matplotlib')